# 02 — Vector Retrieval Debug Notebook

**Day 6** — Explore the LanceDB vector index and `LanceDBVectorSearcher`.

This notebook lets you:
1. Load the embedding model and searcher
2. Run queries and inspect results
3. Compare scores and see what the index returns
4. Peek at the raw LanceDB table
5. Spot-check embedding similarity between queries and chunks

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# If running from notebooks/ dir, add project root to path
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

## 1. Load the embedding model and vector searcher

In [ ]:
from workbench.data_build.embeddings import SentenceTransformerEmbedder
from workbench.retrieval.vector_search import LanceDBVectorSearcher
from workbench.core.types import Query

# Load embedding model
embedder = SentenceTransformerEmbedder(
    model_name="intfloat/multilingual-e5-base",
    device="mps",
)
print(f"Embedder: {embedder}")

# Connect to the LanceDB index
db_path = project_root / "data" / "indexes" / "active" / "lancedb"
searcher = LanceDBVectorSearcher(
    db_path=db_path,
    embedder=embedder,
)
print(f"Searcher: {searcher}")

## 2. Peek at the raw LanceDB table

In [ ]:
import lancedb

db = lancedb.connect(str(db_path))
table = db.open_table("chunks")

print(f"Table name:  chunks")
print(f"Row count:   {table.count_rows():,}")
print(f"Schema:")
print(table.schema)

In [ ]:
# Preview first 5 rows (drop the vector column for readability)
import pandas as pd

sample_df = table.to_pandas()[:5]
display_cols = [c for c in sample_df.columns if c != "vector"]
sample_df[display_cols]

## 3. Run a single query

In [ ]:
query = Query(text="What is the solar system?")
results = searcher.search(query, top_k=5)

print(f"Query: {query.text!r}")
print(f"Results: {len(results)}\n")

for i, r in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(f"  chunk_id: {r.chunk_id}")
    print(f"  score:    {r.score:.4f}")
    print(f"  title:    {r.metadata.get('title', '?')}")
    print(f"  distance: {r.metadata.get('distance', '?'):.4f}")
    print()

## 4. Look at the actual chunk text for the top results

In [ ]:
# Fetch the full text for the top results from LanceDB
top_ids = [r.chunk_id for r in results]

for r in results[:3]:
    # Search by chunk_id in the table
    rows = table.search().where(f"chunk_id = '{r.chunk_id}'").limit(1).to_list()
    if rows:
        row = rows[0]
        text_preview = row["text"][:300]
        print(f"=== {r.chunk_id} (score={r.score:.4f}) ===")
        print(f"Title: {row['title']}")
        print(f"Text:  {text_preview}...")
        print()

## 5. Compare multiple queries side by side

In [ ]:
test_queries = [
    "What is photosynthesis?",
    "Who was Albert Einstein?",
    "History of the Roman Empire",
    "How do computers work?",
    "What causes earthquakes?",
]

for q_text in test_queries:
    q = Query(text=q_text)
    res = searcher.search(q, top_k=3)
    top_titles = [r.metadata.get("title", "?") for r in res]
    top_scores = [f"{r.score:.3f}" for r in res]
    print(f"Q: {q_text}")
    print(f"   Top 3: {list(zip(top_titles, top_scores))}")
    print()

## 6. Embedding similarity deep dive

Let's look at the raw cosine similarity between a query and specific chunks.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Embed a query
query_text = "How do plants convert sunlight into energy?"
query_vec = embedder.embed_query(query_text)

# Get the top 5 results and their vectors from LanceDB
raw_results = table.search(query_vec).limit(5).to_list()

print(f"Query: {query_text!r}\n")
for row in raw_results:
    sim = cosine_similarity(query_vec, row["vector"])
    print(f"  cos_sim={sim:.4f}  dist={row['_distance']:.4f}  title={row['title'][:50]}")

## 7. Score distribution — how spread out are the results?

If your top scores are all clustered together, the index may struggle to
distinguish relevant from irrelevant. A good sign is a clear gap between
the top result and the rest.

In [ ]:
query_text = "History of the United States"
q = Query(text=query_text)
results_20 = searcher.search(q, top_k=20)

scores = [r.score for r in results_20]
ranks = list(range(1, len(scores) + 1))

print(f"Query: {query_text!r}")
print(f"Score range: {min(scores):.4f} — {max(scores):.4f}")
print(f"Score gap (rank 1 vs 2): {scores[0] - scores[1]:.4f}")
print()

# Simple text bar chart
for rank, score in zip(ranks, scores):
    bar_len = int(score * 50)
    title = results_20[rank - 1].metadata.get("title", "?")[:30]
    print(f"  {rank:2d}. {'█' * bar_len} {score:.4f}  {title}")

## 8. Try your own query

Edit the cell below and experiment!

In [ ]:
# ---- Edit this query ----
my_query = "your question here"
my_top_k = 5
# --------------------------

q = Query(text=my_query)
res = searcher.search(q, top_k=my_top_k)

for i, r in enumerate(res, 1):
    # Fetch chunk text
    rows = table.search().where(f"chunk_id = '{r.chunk_id}'").limit(1).to_list()
    text_preview = rows[0]["text"][:200] if rows else "(not found)"
    print(f"--- {i}. score={r.score:.4f}  title={r.metadata.get('title', '?')} ---")
    print(f"    {text_preview}...")
    print()

## 9. Quick stats about the index

In [ ]:
all_data = table.to_pandas()

print(f"Total chunks:     {len(all_data):,}")
print(f"Unique titles:    {all_data['title'].nunique():,}")
print(f"Unique doc IDs:   {all_data['document_id'].nunique():,}")
print(f"Vector dimension: {len(all_data.iloc[0]['vector'])}")
print()

text_lengths = all_data["text"].str.len()
print(f"Chunk text length stats:")
print(f"  min:    {text_lengths.min():,}")
print(f"  median: {text_lengths.median():,.0f}")
print(f"  mean:   {text_lengths.mean():,.0f}")
print(f"  max:    {text_lengths.max():,}")
print()

# Top 10 most-chunked articles
print("Top 10 articles by chunk count:")
top_articles = all_data.groupby("title").size().sort_values(ascending=False).head(10)
for title, count in top_articles.items():
    print(f"  {count:4d} chunks — {title}")